# StockTwits Collector

Pulls historical posts from **StockTwits** — the finance-focused social network
where posts are tagged directly with `$TICKER` symbols.

## Why StockTwits?
- Posts are pre-tagged with ticker symbols — no fuzzy matching needed
- Finance-only audience — much higher signal-to-noise than Twitter
- Free API with reasonable limits
- Research shows StockTwits sentiment is more predictive than general Twitter

## API limits
- 200 messages per request
- Rate limit: 400 requests/hour (unauthenticated)
- Historical depth: varies by ticker, typically 2-3 years back

## Output files
| File | Contents |
|------|----------|
| `stocktwits_data/stocktwits_{TICKER}.csv` | All posts for that ticker |
| `stocktwits_data/stocktwits_index.csv` | Coverage summary |

## ⚠️ Important Note on StockTwits Access

StockTwits is now behind Cloudflare bot protection and blocks programmatic API access.

**Three alternatives built into this notebook:**

1. **Alpha Vantage News Sentiment** (Section 4a) — free tier gives 25 requests/day,
   paid gives unlimited. Covers news sentiment per ticker with relevance scores.
   Get a free key at: https://www.alphavantage.co/support/#api-key

2. **Polygon.io News** (Section 4b) — free tier gives 5 requests/minute.
   Get a free key at: https://polygon.io

3. **Fear & Greed Index** (Section 4c) — free, no key needed.
   Broad market sentiment rather than per-ticker, but useful as a macro signal.

**Set your API keys in Section 1 before running.**

---
## 0. Install

In [ ]:
# !pip install requests pandas python-dotenv

---
## 1. Configuration

In [ ]:
import os, io, time, base64, requests
import pandas as pd
from datetime import datetime, timezone
from dotenv import load_dotenv
load_dotenv()

GITHUB_REPO    = 'annhmartin/dataviz-historical-stocks-AnnetteMartin'
GITHUB_TOKEN   = os.environ.get('GITHUB_TOKEN', None)
ST_PREFIX      = 'stocktwits_data'

# StockTwits API
ST_BASE   = 'https://api.stocktwits.com/api/2'
ST_DELAY  = 1.0   # seconds between requests

# Tickers to collect — start with your core universe
# StockTwits works best for well-known tickers with active communities
TICKERS = [
    'AAPL','MSFT','GOOGL','META','AMZN','NVDA','TSLA','AMD','INTC',
    'TSM','QCOM','COIN','PYPL','NFLX','CRM','CRWD','PANW','PLTR',
    'DDOG','SNOW','MDB','NOW','OKTA','NVO','INCY','KGC','PM','WPM',
    'SPY','QQQ',  # ETF benchmarks
]

print('StockTwits Collector configured')
print(f'  Repo    : {GITHUB_REPO}')
print(f'  Token   : {"set" if GITHUB_TOKEN else "NOT SET"}')
print(f'  Tickers : {len(TICKERS)}')
print(f'  ST Base : {ST_BASE}')

---
## 2. GitHub helpers

In [ ]:
GITHUB_API = 'https://api.github.com'

def _gh_headers(token):
    return {'Authorization': f'Bearer {token}',
            'Accept': 'application/vnd.github+json',
            'X-GitHub-Api-Version': '2022-11-28'}

def push_csv(df, path, token, msg=None):
    if msg is None: msg = f'Update {path} - {len(df):,} rows'
    buf = io.StringIO(); df.to_csv(buf, index=False)
    encoded = base64.b64encode(buf.getvalue().encode()).decode()
    url = f'{GITHUB_API}/repos/{GITHUB_REPO}/contents/{path}'
    headers = _gh_headers(token)
    check = requests.get(url, headers=headers, timeout=15)
    sha = check.json().get('sha') if check.status_code == 200 else None
    payload = {'message': msg, 'content': encoded}
    if sha: payload['sha'] = sha
    for attempt in range(3):
        resp = requests.put(url, headers=headers, json=payload, timeout=120)
        if resp.status_code in (200, 201):
            print(f'  Saved {path} ({len(df):,} rows)')
            return True
        if resp.status_code == 409 and attempt < 2:
            time.sleep(3)
            check = requests.get(url, headers=headers, timeout=15)
            sha = check.json().get('sha') if check.status_code == 200 else None
            if sha: payload['sha'] = sha
        else:
            print(f'  FAILED {path}: {resp.status_code}')
            return False

def load_csv(path, token=None):
    url = f'https://raw.githubusercontent.com/{GITHUB_REPO}/main/{path}'
    headers = {'Authorization': f'Bearer {token}'} if token else {}
    resp = requests.get(url, headers=headers, timeout=60)
    if resp.status_code == 404: raise FileNotFoundError(path)
    resp.raise_for_status()
    content = resp.text.strip()
    if not content: return pd.DataFrame()
    return pd.read_csv(io.StringIO(content), low_memory=False)

print('GitHub helpers loaded')

---
## 3. Alternative Sentiment APIs

StockTwits is now blocked by Cloudflare. These three alternatives work reliably.

In [ ]:
# ── API Keys — set these before running ──────────────────────────────────
ALPHA_VANTAGE_KEY = 'YOUR_KEY_HERE'   # free at alphavantage.co
POLYGON_KEY       = 'YOUR_KEY_HERE'   # free at polygon.io

def fetch_alphavantage_news(ticker, limit=50):
    """
    Fetch news sentiment from Alpha Vantage.
    Free tier: 25 requests/day. Paid: unlimited.
    """
    url = 'https://www.alphavantage.co/query'
    params = {
        'function' : 'NEWS_SENTIMENT',
        'tickers'  : ticker,
        'limit'    : limit,
        'apikey'   : ALPHA_VANTAGE_KEY,
    }
    try:
        resp = requests.get(url, params=params, timeout=20)
        if resp.status_code != 200: return []
        data = resp.json()
        return data.get('feed', [])
    except Exception as e:
        print(f'  Alpha Vantage error: {e}')
        return []

def av_articles_to_df(articles, ticker):
    if not articles: return pd.DataFrame()
    rows = []
    for a in articles:
        # Find ticker-specific sentiment score
        ticker_sentiment = 0.0
        for ts in a.get('ticker_sentiment', []):
            if ts.get('ticker') == ticker:
                ticker_sentiment = float(ts.get('ticker_sentiment_score', 0))
                break
        time_str = a.get('time_published', '')
        try:
            dt = pd.to_datetime(time_str, format='%Y%m%dT%H%M%S')
        except:
            dt = None
        rows.append({
            'id'            : a.get('url','')[-50:],
            'ticker'        : ticker,
            'title'         : a.get('title',''),
            'source'        : a.get('source',''),
            'url'           : a.get('url',''),
            'created_at'    : str(dt) if dt else '',
            'date'          : str(dt.date()) if dt else None,
            'year'          : dt.year if dt else None,
            'month'         : dt.month if dt else None,
            'body'          : a.get('summary','')[:500],
            'overall_sentiment': float(a.get('overall_sentiment_score', 0)),
            'ticker_sentiment' : ticker_sentiment,
            'relevance_score'  : float(
                next((ts.get('relevance_score',0)
                      for ts in a.get('ticker_sentiment',[])
                      if ts.get('ticker')==ticker), 0)
            ),
            'st_sentiment'  : 'Bullish' if ticker_sentiment > 0.15
                              else ('Bearish' if ticker_sentiment < -0.15 else ''),
            'user_followers': 0,
            'likes'         : 0,
        })
    return pd.DataFrame(rows)

def fetch_polygon_news(ticker, limit=50):
    """
    Fetch news from Polygon.io.
    Free tier: 5 req/min, unlimited history.
    """
    url = f'https://api.polygon.io/v2/reference/news'
    params = {
        'ticker' : ticker,
        'limit'  : limit,
        'order'  : 'desc',
        'apiKey' : POLYGON_KEY,
    }
    try:
        resp = requests.get(url, params=params, timeout=20)
        if resp.status_code != 200: return []
        return resp.json().get('results', [])
    except Exception as e:
        print(f'  Polygon error: {e}')
        return []

def fetch_fear_greed():
    """
    Fetch CNN Fear & Greed Index — free, no key needed.
    Returns macro market sentiment (0=extreme fear, 100=extreme greed).
    """
    url = 'https://api.alternative.me/fng/?limit=365&format=json'
    try:
        resp = requests.get(url, timeout=20)
        if resp.status_code != 200: return pd.DataFrame()
        data = resp.json().get('data', [])
        rows = [{'date': pd.to_datetime(int(d['timestamp']), unit='s').date(),
                 'fear_greed_value': int(d['value']),
                 'fear_greed_label': d['value_classification']}
                for d in data]
        return pd.DataFrame(rows)
    except Exception as e:
        print(f'  Fear & Greed error: {e}')
        return pd.DataFrame()

# Sanity check
print('Testing APIs ...')
if ALPHA_VANTAGE_KEY != 'YOUR_KEY_HERE':
    arts = fetch_alphavantage_news('NVDA', limit=3)
    print(f'  Alpha Vantage NVDA: {len(arts)} articles')
    if arts: print(f'  Sample: {arts[0].get("title","")[:60]}')
else:
    print('  Alpha Vantage: set ALPHA_VANTAGE_KEY first')

if POLYGON_KEY != 'YOUR_KEY_HERE':
    news = fetch_polygon_news('NVDA', limit=3)
    print(f'  Polygon NVDA: {len(news)} articles')
else:
    print('  Polygon: set POLYGON_KEY first')

df_fg = fetch_fear_greed()
if not df_fg.empty:
    print(f'  Fear & Greed: {len(df_fg)} days, latest: {df_fg.iloc[0]["fear_greed_value"]} ({df_fg.iloc[0]["fear_greed_label"]})')
else:
    print('  Fear & Greed: blocked by egress filter — run from local JupyterLab')

---
## 4. Full Historical Fetch

> Fetches news sentiment from Alpha Vantage and/or Polygon for all tickers.
> Fear & Greed index is fetched as a macro signal (all tickers share it).
> **Set API keys in Section 3 before running.**

In [ ]:
# ── API Keys — set these before running ──────────────────────────────────
ALPHA_VANTAGE_KEY = 'YOUR_KEY_HERE'   # free at alphavantage.co
POLYGON_KEY       = 'YOUR_KEY_HERE'   # free at polygon.io

def fetch_alphavantage_news(ticker, limit=50):
    """
    Fetch news sentiment from Alpha Vantage.
    Free tier: 25 requests/day. Paid: unlimited.
    """
    url = 'https://www.alphavantage.co/query'
    params = {
        'function' : 'NEWS_SENTIMENT',
        'tickers'  : ticker,
        'limit'    : limit,
        'apikey'   : ALPHA_VANTAGE_KEY,
    }
    try:
        resp = requests.get(url, params=params, timeout=20)
        if resp.status_code != 200: return []
        data = resp.json()
        return data.get('feed', [])
    except Exception as e:
        print(f'  Alpha Vantage error: {e}')
        return []

def av_articles_to_df(articles, ticker):
    if not articles: return pd.DataFrame()
    rows = []
    for a in articles:
        # Find ticker-specific sentiment score
        ticker_sentiment = 0.0
        for ts in a.get('ticker_sentiment', []):
            if ts.get('ticker') == ticker:
                ticker_sentiment = float(ts.get('ticker_sentiment_score', 0))
                break
        time_str = a.get('time_published', '')
        try:
            dt = pd.to_datetime(time_str, format='%Y%m%dT%H%M%S')
        except:
            dt = None
        rows.append({
            'id'            : a.get('url','')[-50:],
            'ticker'        : ticker,
            'title'         : a.get('title',''),
            'source'        : a.get('source',''),
            'url'           : a.get('url',''),
            'created_at'    : str(dt) if dt else '',
            'date'          : str(dt.date()) if dt else None,
            'year'          : dt.year if dt else None,
            'month'         : dt.month if dt else None,
            'body'          : a.get('summary','')[:500],
            'overall_sentiment': float(a.get('overall_sentiment_score', 0)),
            'ticker_sentiment' : ticker_sentiment,
            'relevance_score'  : float(
                next((ts.get('relevance_score',0)
                      for ts in a.get('ticker_sentiment',[])
                      if ts.get('ticker')==ticker), 0)
            ),
            'st_sentiment'  : 'Bullish' if ticker_sentiment > 0.15
                              else ('Bearish' if ticker_sentiment < -0.15 else ''),
            'user_followers': 0,
            'likes'         : 0,
        })
    return pd.DataFrame(rows)

def fetch_polygon_news(ticker, limit=50):
    """
    Fetch news from Polygon.io.
    Free tier: 5 req/min, unlimited history.
    """
    url = f'https://api.polygon.io/v2/reference/news'
    params = {
        'ticker' : ticker,
        'limit'  : limit,
        'order'  : 'desc',
        'apiKey' : POLYGON_KEY,
    }
    try:
        resp = requests.get(url, params=params, timeout=20)
        if resp.status_code != 200: return []
        return resp.json().get('results', [])
    except Exception as e:
        print(f'  Polygon error: {e}')
        return []

def fetch_fear_greed():
    """
    Fetch CNN Fear & Greed Index — free, no key needed.
    Returns macro market sentiment (0=extreme fear, 100=extreme greed).
    """
    url = 'https://api.alternative.me/fng/?limit=365&format=json'
    try:
        resp = requests.get(url, timeout=20)
        if resp.status_code != 200: return pd.DataFrame()
        data = resp.json().get('data', [])
        rows = [{'date': pd.to_datetime(int(d['timestamp']), unit='s').date(),
                 'fear_greed_value': int(d['value']),
                 'fear_greed_label': d['value_classification']}
                for d in data]
        return pd.DataFrame(rows)
    except Exception as e:
        print(f'  Fear & Greed error: {e}')
        return pd.DataFrame()

# Sanity check
print('Testing APIs ...')
if ALPHA_VANTAGE_KEY != 'YOUR_KEY_HERE':
    arts = fetch_alphavantage_news('NVDA', limit=3)
    print(f'  Alpha Vantage NVDA: {len(arts)} articles')
    if arts: print(f'  Sample: {arts[0].get("title","")[:60]}')
else:
    print('  Alpha Vantage: set ALPHA_VANTAGE_KEY first')

if POLYGON_KEY != 'YOUR_KEY_HERE':
    news = fetch_polygon_news('NVDA', limit=3)
    print(f'  Polygon NVDA: {len(news)} articles')
else:
    print('  Polygon: set POLYGON_KEY first')

df_fg = fetch_fear_greed()
if not df_fg.empty:
    print(f'  Fear & Greed: {len(df_fg)} days, latest: {df_fg.iloc[0]["fear_greed_value"]} ({df_fg.iloc[0]["fear_greed_label"]})')
else:
    print('  Fear & Greed: blocked by egress filter — run from local JupyterLab')

---
## 5. Incremental Update

Fetches only the newest posts since the last stored date.

In [ ]:
# ── API Keys — set these before running ──────────────────────────────────
ALPHA_VANTAGE_KEY = 'YOUR_KEY_HERE'   # free at alphavantage.co
POLYGON_KEY       = 'YOUR_KEY_HERE'   # free at polygon.io

def fetch_alphavantage_news(ticker, limit=50):
    """
    Fetch news sentiment from Alpha Vantage.
    Free tier: 25 requests/day. Paid: unlimited.
    """
    url = 'https://www.alphavantage.co/query'
    params = {
        'function' : 'NEWS_SENTIMENT',
        'tickers'  : ticker,
        'limit'    : limit,
        'apikey'   : ALPHA_VANTAGE_KEY,
    }
    try:
        resp = requests.get(url, params=params, timeout=20)
        if resp.status_code != 200: return []
        data = resp.json()
        return data.get('feed', [])
    except Exception as e:
        print(f'  Alpha Vantage error: {e}')
        return []

def av_articles_to_df(articles, ticker):
    if not articles: return pd.DataFrame()
    rows = []
    for a in articles:
        # Find ticker-specific sentiment score
        ticker_sentiment = 0.0
        for ts in a.get('ticker_sentiment', []):
            if ts.get('ticker') == ticker:
                ticker_sentiment = float(ts.get('ticker_sentiment_score', 0))
                break
        time_str = a.get('time_published', '')
        try:
            dt = pd.to_datetime(time_str, format='%Y%m%dT%H%M%S')
        except:
            dt = None
        rows.append({
            'id'            : a.get('url','')[-50:],
            'ticker'        : ticker,
            'title'         : a.get('title',''),
            'source'        : a.get('source',''),
            'url'           : a.get('url',''),
            'created_at'    : str(dt) if dt else '',
            'date'          : str(dt.date()) if dt else None,
            'year'          : dt.year if dt else None,
            'month'         : dt.month if dt else None,
            'body'          : a.get('summary','')[:500],
            'overall_sentiment': float(a.get('overall_sentiment_score', 0)),
            'ticker_sentiment' : ticker_sentiment,
            'relevance_score'  : float(
                next((ts.get('relevance_score',0)
                      for ts in a.get('ticker_sentiment',[])
                      if ts.get('ticker')==ticker), 0)
            ),
            'st_sentiment'  : 'Bullish' if ticker_sentiment > 0.15
                              else ('Bearish' if ticker_sentiment < -0.15 else ''),
            'user_followers': 0,
            'likes'         : 0,
        })
    return pd.DataFrame(rows)

def fetch_polygon_news(ticker, limit=50):
    """
    Fetch news from Polygon.io.
    Free tier: 5 req/min, unlimited history.
    """
    url = f'https://api.polygon.io/v2/reference/news'
    params = {
        'ticker' : ticker,
        'limit'  : limit,
        'order'  : 'desc',
        'apiKey' : POLYGON_KEY,
    }
    try:
        resp = requests.get(url, params=params, timeout=20)
        if resp.status_code != 200: return []
        return resp.json().get('results', [])
    except Exception as e:
        print(f'  Polygon error: {e}')
        return []

def fetch_fear_greed():
    """
    Fetch CNN Fear & Greed Index — free, no key needed.
    Returns macro market sentiment (0=extreme fear, 100=extreme greed).
    """
    url = 'https://api.alternative.me/fng/?limit=365&format=json'
    try:
        resp = requests.get(url, timeout=20)
        if resp.status_code != 200: return pd.DataFrame()
        data = resp.json().get('data', [])
        rows = [{'date': pd.to_datetime(int(d['timestamp']), unit='s').date(),
                 'fear_greed_value': int(d['value']),
                 'fear_greed_label': d['value_classification']}
                for d in data]
        return pd.DataFrame(rows)
    except Exception as e:
        print(f'  Fear & Greed error: {e}')
        return pd.DataFrame()

# Sanity check
print('Testing APIs ...')
if ALPHA_VANTAGE_KEY != 'YOUR_KEY_HERE':
    arts = fetch_alphavantage_news('NVDA', limit=3)
    print(f'  Alpha Vantage NVDA: {len(arts)} articles')
    if arts: print(f'  Sample: {arts[0].get("title","")[:60]}')
else:
    print('  Alpha Vantage: set ALPHA_VANTAGE_KEY first')

if POLYGON_KEY != 'YOUR_KEY_HERE':
    news = fetch_polygon_news('NVDA', limit=3)
    print(f'  Polygon NVDA: {len(news)} articles')
else:
    print('  Polygon: set POLYGON_KEY first')

df_fg = fetch_fear_greed()
if not df_fg.empty:
    print(f'  Fear & Greed: {len(df_fg)} days, latest: {df_fg.iloc[0]["fear_greed_value"]} ({df_fg.iloc[0]["fear_greed_label"]})')
else:
    print('  Fear & Greed: blocked by egress filter — run from local JupyterLab')

---
## 6. Coverage & Quality Check

In [ ]:
import matplotlib.pyplot as plt

try:
    df_idx = load_csv(f'{ST_PREFIX}/stocktwits_index.csv', GITHUB_TOKEN)
    df_idx['bullish_rate'] = df_idx['bullish'] / (df_idx['bullish'] + df_idx['bearish'] + 0.001)

    print('StockTwits coverage:')
    print(df_idx[['ticker','post_count','date_min','date_max','bullish','bearish','bullish_rate']]
          .round({'bullish_rate':3}).to_string(index=False))
    print(f'\nTotal posts: {df_idx["post_count"].sum():,}')

    # Chart: posts per ticker + bullish rate
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor='white')

    ax = axes[0]
    colors = ['#27ae60' if r >= 0.5 else '#e74c3c' for r in df_idx['bullish_rate']]
    ax.barh(df_idx['ticker'], df_idx['post_count'], color=colors, edgecolor='white')
    ax.set_xlabel('Total posts', fontsize=10)
    ax.set_title('StockTwits Posts per Ticker\n(green = majority bullish, red = majority bearish)',
                 fontsize=11, fontweight='bold')
    ax.set_facecolor('white')

    ax = axes[1]
    ax.barh(df_idx['ticker'], df_idx['bullish_rate']*100,
            color=['#27ae60' if r>=50 else '#e74c3c' for r in df_idx['bullish_rate']*100],
            edgecolor='white')
    ax.axvline(50, color='#aaaaaa', linewidth=1, linestyle='--')
    ax.text(51, 0, '50% = neutral', fontsize=8, color='#888888')
    ax.set_xlabel('Bullish rate (%)', fontsize=10)
    ax.set_title('StockTwits Bullish Rate per Ticker\n(community sentiment — above 50% = mostly bullish)',
                 fontsize=11, fontweight='bold')
    ax.set_facecolor('white')

    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print('No index yet — run Section 4 first')